# Data preprocessing — California single-family price prediction

Goal: predict close price for a California single-family address at prediction time, including homes that are off market. Use property characteristics and location that can be obtained for that address. Sold-record attributes must be checked against the eventual prediction-time data source.

Deliverables: this notebook and `data/cleaned_single_family_sales.csv`. The cleaned table retains original units and missing features. Training-only preprocessing handles imputation and scaling. Export does not depend on choosing a model or training window.

Reserve the most recent available sale month for testing. Tune training-window length using earlier validation months, never the test month. Report gaps and whether the newest month is incomplete.

Input context: six historical monthly CRMLS exports, November 2025 through April 2026. The provider background does not establish whether repeated keys represent corrections or distinct sale events.


## Setup and configuration

Run cells in order. Configuration declares the current rules; inputs remain unchanged in `raw`.


In [ ]:
# Standard-library imports keep configuration and run metadata portable.
import hashlib
import json
import platform
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Keep cleaning/model choices together so each run uses an explicit policy.
NUMERIC_CANDIDATES = [
    "LivingArea", "LotSizeSquareFeet", "BedroomsTotal", "BathroomsTotalInteger",
    "GarageSpaces", "ParkingTotal", "Stories", "YearBuilt", "Latitude", "Longitude",
]
CATEGORICAL_CANDIDATES = ["City", "PostalCode"]
REQUIRED_COLUMNS = {
    "PropertyType", "PropertySubType", "StateOrProvince",
    "ClosePrice", "CloseDate", "ListingKeyNumeric",
}
TEXT_COLUMNS = {
    "ListingKeyNumeric": "string", "ListingKey": "string",
    "ListingId": "string", "PostalCode": "string",
}
COORDINATE_BOUNDS = {"Latitude": (-90, 90), "Longitude": (-180, 180)}
CANDIDATE_WINDOWS = [1, 3, 6, 12]
# With April reserved for testing, use February and March as shared folds.
VALIDATION_OFFSETS = [2, 1]
RIDGE_ALPHA = 1.0
# Conservative baseline screening assumptions, not physical or CRMLS limits.
MEASUREMENT_SCREENING = {
    "minimum_positive_lot_sqft": 1.0,
    "suspected_double_conversion_minimum_area": 1000.0,
    "garage_review_limit": 20.0,
    "parking_review_limit": 50.0,
}
SQFT_PER_ACRE = 43560.0
DUPLICATE_POLICY = "quarantine_all_conflicting_keys"
# Manually reviewed sale-price anomalies. No numeric corrections are inferred.
PRICE_REVIEW_REASONS = {
    "1144223437": "$1.75 close price versus $1,749,000 list price",
    "1110035161": "$8,300 close price versus $2,000,000 list price",
    "1138900104": "$31,000 close price versus $310,000 list price",
    "1126691917": "$37,000 close price versus $385,000 list price",
    "1149697906": "$796,000,000 close price versus $824,900 list price",
    "1150712418": "$768,500,000 close price versus $750,000 list price",
    "1126046881": "$740,000,000 close price versus $735,000 list price",
    "1148201765": "$664,250,000 close price versus $750,000 list price",
    "1153305774": "$645,000,000 close price versus $645,000 list price",
    "1135757473": "$472,000,000 close price versus $479,000 list price",
    "1111473680": "$199,947,850 close price versus $1,499,000 list price",
}

# Resolve paths from either the project root or the notebooks directory.
project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "notebooks" / "01_exploration.ipynb").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the kernel in the project root or notebooks directory.")
data_dir = project_root / "data"


def file_sha256(path):
    """Fingerprint an input or artifact without loading the whole file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def audit_step(step, before, after):
    """Record actual row removals; validity checks may count overlapping issues."""
    row_audit.append({
        "step": step, "rows_remaining": int(after), "rows_removed": int(before - after),
    })


## Import monthly source files


In [ ]:
# Sort file paths to make import order deterministic. Exclude generated CSVs.
files = sorted(
    path for path in data_dir.glob("*")
    if path.is_file() and path.suffix.lower() == ".csv"
    and path.name.lower().startswith("crmlssold")
)
if not files:
    raise FileNotFoundError(f"No CRMLSSold CSVs found in {data_dir}")

# Validate each month's schema, rather than hiding omissions in concatenation.
frames = []
input_manifest = []
for path in files:
    frame = pd.read_csv(path, low_memory=False, dtype=TEXT_COLUMNS)
    missing_columns = REQUIRED_COLUMNS - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{path.name}: required columns missing: {sorted(missing_columns)}")
    frames.append(frame.assign(source_file=path.name))
    input_manifest.append({
        "file": path.name, "rows": len(frame), "sha256": file_sha256(path),
    })
raw = pd.concat(frames, ignore_index=True)
del frames, frame  # Release references to the individual imports after concatenation.
print(f"Imported {len(raw):,} rows from {len(files)} files")
print("\nInput files — row counts and content fingerprints:")
display(pd.DataFrame(input_manifest))


## 1. California scope, target validity, and duplicates

Keep residential single-family sales with a normalized state value of `CA` or `California`. Exclude and count missing or other states. A California state label establishes scope; it does not independently verify coordinates.

Exclude nonpositive/nonfinite prices, unparseable sale dates, and missing/blank listing keys. Remove exact repeats ignoring source provenance. **Set aside every record belonging to a conflicting listing key** in a separate review table. Do not infer that the newest monthly file contains a correction. Export the review records separately so they can be verified and restored later.

Review of the supplied outputs found 39 conflicting keys covering 79 records: all have different close dates and 12 have different close prices. This is an observation from the current inputs, not a fixed row-count requirement. Exclusion may affect sample coverage; record actual counts on every run.

Validity counts are measured before exclusion and can overlap. The audit reports actual row removals. Positive price extremes remain for review; passing these checks does not prove every value is accurate.


In [ ]:
# Reset the audit when this cell is rerun; always start from the unchanged import.
row_audit = [{"step": "imported", "rows_remaining": len(raw), "rows_removed": 0}]

# 1a. Scope: California residential single-family records.
state = raw["StateOrProvince"].astype("string").str.strip().str.upper()
print("\nState / province counts before California filtering (includes missing values):")
display(state.value_counts(dropna=False).rename_axis("State / province").rename("Row count").to_frame())
cleaned = raw.loc[
    raw["PropertyType"].eq("Residential") &
    raw["PropertySubType"].eq("SingleFamilyResidence") &
    state.isin(["CA", "CALIFORNIA"])
].copy()
audit_step("California residential single-family scope", len(raw), len(cleaned))
cleaned["StateOrProvince"] = "CA"
cleaned["ListingKeyNumeric"] = cleaned["ListingKeyNumeric"].astype("string").str.strip().replace("", pd.NA)
# 1b. Validate target, sale date, and listing identity before excluding rows.
cleaned["ClosePrice"] = pd.to_numeric(cleaned["ClosePrice"], errors="coerce")
cleaned["CloseDate"] = pd.to_datetime(cleaned["CloseDate"], errors="coerce")
print("Rows with missing, nonnumeric, nonfinite, or nonpositive close prices:", (~(cleaned["ClosePrice"].gt(0) & np.isfinite(cleaned["ClosePrice"]))).sum())
print("Rows with missing or unparseable close dates:", cleaned["CloseDate"].isna().sum())
print("Rows with missing or blank listing keys:", cleaned["ListingKeyNumeric"].isna().sum())
before = len(cleaned)
cleaned = cleaned.loc[
    cleaned["ClosePrice"].gt(0) & np.isfinite(cleaned["ClosePrice"]) &
    cleaned["CloseDate"].notna() & cleaned["ListingKeyNumeric"].notna()
].copy()
audit_step("valid price, date, and listing key", before, len(cleaned))
# 1c. Remove exact repeats, then inspect differing records sharing a key.
record_columns = [c for c in cleaned if c != "source_file"]
before = len(cleaned)
cleaned = cleaned.drop_duplicates(subset=record_columns)
audit_step("exact duplicates", before, len(cleaned))
print("Exact duplicate rows removed (source file ignored):", before - len(cleaned))
conflicts = cleaned.loc[cleaned.duplicated("ListingKeyNumeric", keep=False)]
if not conflicts.empty:
    print("Listing keys with conflicting records:", conflicts["ListingKeyNumeric"].nunique())
    # Match current model candidates; include sale identity and scope fields.
    review_columns = [
        col for col in ["ClosePrice", "CloseDate", "PropertyType", "PropertySubType",
                       "StateOrProvince", *NUMERIC_CANDIDATES, *CATEGORICAL_CANDIDATES]
        if col in cleaned
    ]
    conflict_summary_rows = []
    important_difference_rows = []
    for key, records in conflicts.groupby("ListingKeyNumeric", sort=True):
        changed = [col for col in record_columns
                   if records[col].nunique(dropna=False) > 1]
        important_changed = [col for col in review_columns if col in changed]
        conflict_summary_rows.append({
            "Listing key": key,
            "Record count": len(records),
            "Source files": ", ".join(sorted(records["source_file"].unique())),
            "Changed columns": ", ".join(changed),
            "Changed target / date / scope / predictors": ", ".join(important_changed) or "None",
        })
        for col in important_changed:
            for _, record in records.iterrows():
                important_difference_rows.append({
                    "Listing key": key,
                    "Field": col,
                    "Source file": record["source_file"],
                    "Value": str(record[col]) if pd.notna(record[col]) else "<missing>",
                })
    conflict_summary = pd.DataFrame(conflict_summary_rows)
    important_differences = pd.DataFrame(important_difference_rows)
    print("\nDuplicate review — all conflicting keys and the columns that differ:")
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None):
        display(conflict_summary)
        print("\nDifferences affecting the target, sale date, scope, or current model candidates:")
        if important_differences.empty:
            print("None. All conflicts are confined to fields outside the current model inputs and sale identity checks.")
        else:
            display(important_differences)
    # Every version is retained separately for review; no sale version is assumed correct.
    print(f"\nSetting aside {len(conflicts):,} records across "
          f"{conflicts['ListingKeyNumeric'].nunique():,} conflicting keys.")

# Keep a separate review table even when a future run has no conflicting records.
duplicate_review = conflicts.copy()
duplicate_review["review_reason"] = "Different records share the same listing key"
before = len(cleaned)
cleaned = cleaned.loc[~cleaned["ListingKeyNumeric"].isin(conflicts["ListingKeyNumeric"])].copy()
audit_step("conflicting listing keys set aside for review", before, len(cleaned))
cleaned = cleaned.reset_index(drop=True)
assert not cleaned.empty, "No usable rows remain."

print("\nCleaning audit — rows remaining and removed at each step:")
display(pd.DataFrame(row_audit).rename(columns={
    "step": "Cleaning step", "rows_remaining": "Rows remaining", "rows_removed": "Rows removed"
}))
print("\nClose-price summary after scope, validity, and duplicate checks (original currency units):")
display(cleaned["ClosePrice"].describe(percentiles=[0.01, 0.5, 0.99])
        .rename_axis("Statistic").rename("Close price").to_frame())


### Section 1 findings and remaining review

The reviewed run imported **122,756 records**, retained **60,694 California residential single-family records**, and set aside **79 records across 39 conflicting listing keys**, leaving **60,615 records** for feature preprocessing. The scoped records had no invalid/nonpositive prices, unparseable sale dates, missing listing keys, or exact duplicates under the checks above. The 62,062 scope exclusions combine property-type, subtype, and state criteria; they are not all out-of-state records.

The retained median close price was **$880,000**, with the 1st and 99th percentiles at **$230,000** and **$6,573,600**. However, the minimum of **$1.75** and maximum of **$796 million** require investigation. A positive price passes the validity check without establishing that it is a plausible single-property sale. Do not automatically remove all observations outside the percentile range or assume these extremes are valid.

**Next:** proceed to section 2 to inspect feature availability, missingness, and impossible property values. Review the extreme-price records and geographic inconsistencies before treating the CSV as a finalized cleaning deliverable. Conflicting-key records remain available for separate verification.

These figures summarize the output reviewed during this session. After changing inputs or cleaning rules, refresh this narrative from the current audit and descriptive statistics.


## 2. Feature selection and missing-value plan

Initial candidates are property characteristics and location. Confirm that the eventual address lookup supplies these fields at prediction time, including for off-market homes. Listing prices, days on market, buyer details, and sale status are not predictors. Exclude identifiers, provenance, target, final status, and sale date from model inputs. Association fees are deferred until billing periods are standardized.

Use median imputation plus missing indicators for numeric fields; use a missing category plus one-hot encoding for categorical fields. Drop features entirely missing in each training fold. Standard scaling suits the Ridge baseline; tree models may not need it. Do not cap legitimate extremes just because plots capped them.

Review geographic flags from exploration before export. Section 1 excludes non-California state labels. Coordinates outside California and plausible extremes still need review; global latitude/longitude bounds only detect impossible coordinates.


In [ ]:
# Convert features without learning any values from the full dataset.
numeric_features = [c for c in NUMERIC_CANDIDATES if c in cleaned]
categorical_features = [c for c in CATEGORICAL_CANDIDATES if c in cleaned]
for c in numeric_features:
    cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
for c, bounds in COORDINATE_BOUNDS.items():
    if c in numeric_features:
        cleaned.loc[~cleaned[c].between(*bounds), c] = np.nan
for c in categorical_features:
    cleaned[c] = cleaned[c].map(lambda v: str(v).strip() if pd.notna(v) and str(v).strip() else np.nan)
feature_columns = numeric_features + categorical_features
assert feature_columns, "Select at least one usable feature."
# Counts and percentages make missingness readable without interpreting fractions.
feature_missingness = pd.DataFrame({
    "Missing rows": cleaned[feature_columns].isna().sum(),
    "Available rows": cleaned[feature_columns].notna().sum(),
    "Missing (%)": cleaned[feature_columns].isna().mean().mul(100),
}).sort_values("Missing (%)", ascending=False, kind="stable")
feature_missingness.index.name = "Feature"
print("\nFeature missingness after type conversion and coordinate validity checks:")
display(feature_missingness.style.format({"Missing rows": "{:,.0f}",
                                         "Available rows": "{:,.0f}",
                                         "Missing (%)": "{:.2f}%"}))


### Review measurements and price extremes

The reviewed missingness output supports retaining all current candidates provisionally: Stories is missing in 10.28% of records, GarageSpaces in 3.86%, and each other feature in less than 2%. Missing values remain in the CSV; training folds supply numeric medians and the categorical missing label.

Missingness alone does not establish validity. The checks below flag nonpositive area/story measurements, negative or fractional counts, and questionable construction years. Flags are for review and do not change values. Zero parking/garage counts can be meaningful, and half-stories can be valid. Inspect the numeric distributions and lowest/highest prices before finalizing rules. Refresh this narrative if the inputs change.


In [ ]:
# Review flags do not change values or remove rows. Decide rules after inspection.
validity_checks = {}
for col in ["LivingArea", "LotSizeSquareFeet", "Stories"]:
    if col in numeric_features:
        validity_checks[f"{col}: zero or negative"] = cleaned[col].le(0)
for col in ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        validity_checks[f"{col}: negative"] = cleaned[col].lt(0)
        validity_checks[f"{col}: fractional count (review)"] = (
            cleaned[col].notna() & cleaned[col].mod(1).ne(0)
        )
# A half-story can be meaningful; do not apply an integer rule to Stories.
if "YearBuilt" in numeric_features:
    validity_checks["YearBuilt: zero or negative"] = cleaned["YearBuilt"].le(0)
    validity_checks["YearBuilt: later than sale year (review)"] = (
        cleaned["YearBuilt"].gt(cleaned["CloseDate"].dt.year)
    )

validity_summary = pd.DataFrame([
    {"Review check": label, "Flagged rows": int(mask.sum())}
    for label, mask in validity_checks.items()
])
print("\nProperty-value review checks (flags can overlap; zeros in count fields are retained):")
display(validity_summary)

# Show actual flagged measurements, rather than a wide table of unrelated fields.
flagged_measurements = pd.concat([
    cleaned.loc[mask, ["ListingKeyNumeric", "CloseDate", *numeric_features]]
           .assign(review_check=label)
    for label, mask in validity_checks.items()
], ignore_index=True) if validity_checks else pd.DataFrame()
if not flagged_measurements.empty:
    print("\nFlagged property measurements — first 30 rows; full table is in flagged_measurements:")
    with pd.option_context("display.max_columns", None):
        display(flagged_measurements.head(30))

print("\nNumeric feature distribution summary — original units, no clipping:")
display(cleaned[numeric_features].describe(percentiles=[0.01, 0.5, 0.99]).T)

# Extreme positive prices passed section 1; inspect them before choosing any exclusions.
price_review_columns = ["ListingKeyNumeric", "CloseDate", "ClosePrice", *feature_columns]
print("\nLowest 10 close prices — retained for review:")
with pd.option_context("display.max_columns", None):
    display(cleaned.nsmallest(10, "ClosePrice")[price_review_columns])
    print("\nHighest 10 close prices — retained for review:")
    display(cleaned.nlargest(10, "ClosePrice")[price_review_columns])


### Investigate possible units and recording errors

The reviewed output contains 18 nonpositive living areas, 43 nonpositive lot areas, 8 negative parking counts, 94 fractional values in each parking/garage field, and 6 construction years later than the sale year. Counts can overlap. The largest lot/parking values, tiny positive lot measurements, and extreme sale prices need source-context review before selecting rules.

Compare lot measurements across available source columns and inspect fractional parking values directly. Missing future construction years may reflect planned completion, so they are not automatically invalid. Listing prices can provide supporting context but cannot establish the correct close price. No multiplying, dividing, rounding, or coordinate sign correction is performed here. These diagnostic fields remain excluded from model inputs.


In [ ]:
# Compare suspicious values with related source fields. These are review aids,
# not added model inputs, and no replacement values are inferred automatically.
context_columns = [col for col in [
    "ListingKeyNumeric", "source_file", "CloseDate", "ClosePrice", "City", "PostalCode",
    "LivingArea", "LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea", "LotSizeUnits",
    "GarageSpaces", "ParkingTotal", "YearBuilt", "NewConstructionYN",
    "Latitude", "Longitude", "OriginalListPrice", "ListPrice",
] if col in cleaned]

print("\nFractional garage / parking values — first 30 records:")
fractional_mask = pd.Series(False, index=cleaned.index)
for col in ["GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        fractional_mask |= cleaned[col].notna() & cleaned[col].mod(1).ne(0)
display(cleaned.loc[fractional_mask, context_columns].head(30))

if "YearBuilt" in numeric_features:
    print("\nConstruction year after sale year — all flagged records:")
    display(cleaned.loc[cleaned["YearBuilt"].gt(cleaned["CloseDate"].dt.year), context_columns])

if "LotSizeSquareFeet" in numeric_features:
    print("\nSmallest positive lot measurements — compare with source lot fields:")
    small_lots = cleaned.loc[cleaned["LotSizeSquareFeet"].gt(0)].nsmallest(10, "LotSizeSquareFeet")
    display(small_lots[context_columns])
    print("\nLargest lot measurements — compare with source lot fields:")
    display(cleaned.nlargest(10, "LotSizeSquareFeet")[context_columns])

for col in ["GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        print(f"\nLargest {col} values — first 10 records:")
        display(cleaned.nlargest(10, col)[context_columns])

# Zero coordinates or a positive longitude are inconsistent with a California
# address. Flag both coordinates together; never silently flip a sign.
if {"Latitude", "Longitude"}.issubset(numeric_features):
    coordinate_review_mask = (
        cleaned["Latitude"].eq(0) | cleaned["Longitude"].ge(0)
    )
    print("\nCalifornia coordinate review — zero latitude or nonnegative longitude:")
    print("Flagged records:", int(coordinate_review_mask.sum()))
    display(cleaned.loc[coordinate_review_mask, context_columns].head(30))

print("\nLowest 10 prices with listing-price context (context does not verify a sale price):")
display(cleaned.nsmallest(10, "ClosePrice")[context_columns])
print("\nHighest 10 prices with listing-price context (context does not verify a sale price):")
display(cleaned.nlargest(10, "ClosePrice")[context_columns])


### Apply measurement-validity rules and provisional screening

Treat nonpositive areas and negative counts as missing. Also treat the following unverified measurements as missing while preserving the property rows and original measurements in the audit:

- Positive lots smaller than 1 square foot.
- Suspected double conversion: source `LotSizeArea` is at least 1,000, approximately equals `LotSizeAcres`, and `LotSizeSquareFeet` approximately equals that value multiplied by 43,560. No corrected value is inferred.
- Garage values above 20 or total parking above 50. These are conservative baseline screening assumptions, not physical limits or CRMLS standards. Valid large-capacity properties may be affected; review the audit and reconsider these assumptions when documentation becomes available.

Retain zeros in count fields, fractional garage/parking values, and the reviewed future construction years. Treat both coordinates as missing when latitude is zero or longitude is nonnegative. Other location discrepancies remain for review. No rounding, inferred price correction, or coordinate sign correction is applied.


In [ ]:
# Apply only established validity rules; keep a record of every modified field.
measurement_change_frames = []
measurement_rule_counts = []
for col in numeric_features:
    if col in {"LivingArea", "LotSizeSquareFeet"}:
        invalid = cleaned[col].le(0)
        reason = "Nonpositive area treated as missing"
    elif col in {"BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"}:
        invalid = cleaned[col].lt(0)
        reason = "Negative count treated as missing"
    else:
        continue
    measurement_rule_counts.append({"Feature": col, "Rule": reason,
                                    "Values changed": int(invalid.sum())})
    if invalid.any():
        changes = cleaned.loc[invalid, ["ListingKeyNumeric", "source_file", col]].copy()
        changes = changes.rename(columns={col: "original_value"})
        changes["feature"] = col
        changes["rule"] = reason
        measurement_change_frames.append(changes)
        cleaned.loc[invalid, col] = np.nan
# Screen unverified positive measurements without inventing replacements.
# Limits are explicit baseline assumptions; large valid properties may be affected.
positive_review_masks = {}
if "LotSizeSquareFeet" in numeric_features:
    lot = cleaned["LotSizeSquareFeet"]
    positive_review_masks[("LotSizeSquareFeet", "Positive lot below 1 square foot; unit unverified")] = (
        lot.gt(0) & lot.lt(MEASUREMENT_SCREENING["minimum_positive_lot_sqft"])
    )
    if {"LotSizeArea", "LotSizeAcres"}.issubset(cleaned.columns):
        source_area = pd.to_numeric(cleaned["LotSizeArea"], errors="coerce")
        source_acres = pd.to_numeric(cleaned["LotSizeAcres"], errors="coerce")
        suspected_conversion = (
            source_area.ge(MEASUREMENT_SCREENING["suspected_double_conversion_minimum_area"])
            & np.isfinite(source_area) & np.isfinite(source_acres) & np.isfinite(lot)
            & np.isclose(source_area, source_acres, rtol=1e-6, atol=0.01)
            & np.isclose(lot, source_area * SQFT_PER_ACRE, rtol=1e-6, atol=1.0)
        )
        positive_review_masks[("LotSizeSquareFeet", "Source area/acres repeat and appear converted again; units unverified")] = suspected_conversion
for col, limit_key in [("GarageSpaces", "garage_review_limit"), ("ParkingTotal", "parking_review_limit")]:
    if col in numeric_features:
        limit = MEASUREMENT_SCREENING[limit_key]
        positive_review_masks[(col, f"Above provisional review limit of {limit:g}; value unverified")] = cleaned[col].gt(limit)
for (col, reason), flagged in positive_review_masks.items():
    # Count changes only where the measurement is still present.
    changed = flagged & cleaned[col].notna()
    measurement_rule_counts.append({"Feature": col, "Rule": reason,
                                    "Values changed": int(changed.sum())})
    if changed.any():
        changes = cleaned.loc[changed, ["ListingKeyNumeric", "source_file", col]].copy()
        changes = changes.rename(columns={col: "original_value"})
        changes["feature"] = col
        changes["rule"] = reason
        measurement_change_frames.append(changes)
        cleaned.loc[changed, col] = np.nan

# Invalidate a demonstrably incorrect coordinate pair together. A sign change
# could still point to the wrong city, so do not infer replacement coordinates.
if {"Latitude", "Longitude"}.issubset(numeric_features):
    invalid_pair = cleaned["Latitude"].eq(0) | cleaned["Longitude"].ge(0)
    print("Invalid California coordinate pairs treated as missing:", int(invalid_pair.sum()))
    for col in ["Latitude", "Longitude"]:
        changed = invalid_pair & cleaned[col].notna()
        reason = "Zero latitude or nonnegative longitude invalidates California coordinate pair"
        measurement_rule_counts.append({"Feature": col, "Rule": reason,
                                        "Values changed": int(changed.sum())})
        if changed.any():
            changes = cleaned.loc[changed, ["ListingKeyNumeric", "source_file", col]].copy()
            changes = changes.rename(columns={col: "original_value"})
            changes["feature"] = col
            changes["rule"] = reason
            measurement_change_frames.append(changes)
        cleaned.loc[invalid_pair, col] = np.nan

measurement_changes = (
    pd.concat(measurement_change_frames, ignore_index=True)
    if measurement_change_frames else pd.DataFrame(columns=[
        "ListingKeyNumeric", "source_file", "original_value", "feature", "rule"
    ])
)
print("\nApplied measurement rules — field changes, not row removals:")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(measurement_rule_counts))
print("Property rows retained:", len(cleaned))

# Refresh missingness after the newly invalidated measurements.
feature_missingness = pd.DataFrame({
    "Missing rows": cleaned[feature_columns].isna().sum(),
    "Available rows": cleaned[feature_columns].notna().sum(),
    "Missing (%)": cleaned[feature_columns].isna().mean().mul(100),
}).sort_values("Missing (%)", ascending=False, kind="stable")
feature_missingness.index.name = "Feature"
print("\nFeature missingness after measurement cleaning:")
display(feature_missingness.style.format({"Missing rows": "{:,.0f}",
                                         "Available rows": "{:,.0f}",
                                         "Missing (%)": "{:.2f}%"}))


### Set aside manually reviewed sale-price anomalies

Exclude the 11 reviewed listing keys from the baseline cleaned table and keep their records in `suspicious_sale_prices_for_review.csv`. These are record-specific review decisions, not a universal price threshold or a rule fitted on held-out model performance. Do not replace sale prices with listing prices or infer a scaling correction. The six low-price and three high-price records outside this review list in the displayed extremes remain provisionally retained.

This review included records from the eventual test month. Document these cleaning decisions before evaluation and freeze the rules; the holdout is reserved from model tuning, but was not hidden from data-quality inspection. The review list is specific to the current exports and needs reassessment for new inputs.


In [ ]:
# Exclude only explicitly reviewed records; retain full originals in raw.
price_review_mask = cleaned["ListingKeyNumeric"].isin(PRICE_REVIEW_REASONS)
price_review = cleaned.loc[price_review_mask].copy()
price_review["review_reason"] = price_review["ListingKeyNumeric"].map(PRICE_REVIEW_REASONS)
before = len(cleaned)
cleaned = cleaned.loc[~price_review_mask].copy()
audit_step("manually reviewed suspicious sale prices set aside", before, len(cleaned))
print(f"Set aside {len(price_review):,} suspicious-price records.")
print("Reviewed keys absent from the current cleaned table:",
      sorted(set(PRICE_REVIEW_REASONS) - set(price_review["ListingKeyNumeric"])))
assert not cleaned.empty, "No records remain after price review."
assert not cleaned["ListingKeyNumeric"].isin(PRICE_REVIEW_REASONS).any()
print("\nUpdated row audit after price review:")
display(pd.DataFrame(row_audit))
print("\nRetained close-price summary after excluding reviewed anomalies:")
display(cleaned["ClosePrice"].describe(percentiles=[0.01, 0.5, 0.99]).to_frame())


### Section 2 decisions and limitations

Retain all current property/location candidates. Convert invalid or explicitly screened measurements to missing instead of dropping property rows. Preserve original changed values in `measurement_cleaning_audit.csv`. Keep fractional garage/parking values without rounding: the supplied documentation does not establish their reporting convention. Retain the reviewed future construction years provisionally, including one record without supporting new-construction status.

Suspected lot-unit errors are not repaired through guessed conversions. Provisional garage/parking screening may mask valid unusually large facilities. The checks do not detect every unit error or validate every coordinate against a California boundary or address. Manually reviewed price exclusions are specific to this dataset. These uncertainties, feature availability for off-market homes, and training only on sold properties limit what the model can establish about any California address.

Missing features remain in the human-readable cleaned CSV. Numeric median imputation, missing indicators, categorical encoding, and scaling are fitted only on each training fold. The following output refreshes row counts, missingness, and field-change counts after all section 2 decisions. It reflects a baseline cleaning policy, not verified accuracy of every source record.


In [ ]:
# Final pre-export summary: use current rows after price review, avoiding stale counts.
feature_missingness = pd.DataFrame({
    "Missing rows": cleaned[feature_columns].isna().sum(),
    "Available rows": cleaned[feature_columns].notna().sum(),
    "Missing (%)": cleaned[feature_columns].isna().mean().mul(100),
}).sort_values("Missing (%)", ascending=False, kind="stable")
feature_missingness.index.name = "Feature"
print(f"\nSection 2 complete — {len(cleaned):,} property rows available for baseline export.")
print("Fractional garage/parking values retained; no rounding or inferred unit corrections.")
print("\nFinal feature missingness after measurement cleaning and price review:")
display(feature_missingness.style.format({"Missing rows": "{:,.0f}",
                                         "Available rows": "{:,.0f}",
                                         "Missing (%)": "{:.2f}%"}))
print("\nMeasurement changes — counts include changes to rows later set aside for price review:")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(measurement_rule_counts))

# Guard the explicit rules before writing files; these checks do not verify source truth.
for col in ["LivingArea", "LotSizeSquareFeet"]:
    if col in numeric_features:
        assert cleaned[col].dropna().gt(0).all(), f"Nonpositive {col} remains."
for col in ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        assert cleaned[col].dropna().ge(0).all(), f"Negative {col} remains."
for col, limit_key in [("GarageSpaces", "garage_review_limit"), ("ParkingTotal", "parking_review_limit")]:
    if col in numeric_features:
        assert cleaned[col].dropna().le(MEASUREMENT_SCREENING[limit_key]).all()
if "LotSizeSquareFeet" in numeric_features:
    assert cleaned["LotSizeSquareFeet"].dropna().ge(MEASUREMENT_SCREENING["minimum_positive_lot_sqft"]).all()


## Cleaned CSV export

Run after reviewing the scope audit, duplicate checks, and feature missingness above. Export property/location candidates, target, sale date, and audit identifiers; omit agent/contact fields and other unused raw columns. Missing features are intentional and are imputed only within training folds. No percentile clipping is applied. This table is a baseline cleaning deliverable; positive price extremes and geographic discrepancies still require review.

The later modeling cells create a separate file with selected training-window membership. They do not overwrite this cleaned table.

Also save `conflicting_listing_records_for_review.csv` and `preprocessing_metadata.json` with content fingerprints, package versions, cleaning rules, and row counts. Outputs are replaced when this cell is rerun. Reproduce a run using identical inputs, notebook code, and recorded package versions.


In [ ]:
audit_columns = ["ListingKeyNumeric", "StateOrProvince", "CloseDate", "source_file"]
export_columns = audit_columns + ["ClosePrice"] + feature_columns
cleaned_export = cleaned[export_columns].copy()
assert not cleaned_export.empty
assert cleaned_export["ListingKeyNumeric"].is_unique
assert cleaned_export["StateOrProvince"].eq("CA").all()
assert (cleaned_export["ClosePrice"].gt(0) & np.isfinite(cleaned_export["ClosePrice"])).all()
assert cleaned_export["CloseDate"].notna().all()
cleaned_csv_path = data_dir / "cleaned_single_family_sales.csv"
# Write stable row order and date formatting for reproducible comparisons.
cleaned_export = cleaned_export.sort_values(
    ["CloseDate", "ListingKeyNumeric"], kind="stable"
)
cleaned_export.to_csv(cleaned_csv_path, index=False, date_format="%Y-%m-%d")

# Review artifacts exclude contact details; the full conflicts remain in memory.
review_columns = [
    col for col in [*audit_columns, "PropertyType", "PropertySubType", "ClosePrice",
                   *feature_columns, "review_reason"] if col in duplicate_review
]
duplicate_review_path = data_dir / "conflicting_listing_records_for_review.csv"
duplicate_review[review_columns].sort_values(
    ["ListingKeyNumeric", "source_file"], kind="stable"
).to_csv(duplicate_review_path, index=False, date_format="%Y-%m-%d")

# Preserve manually reviewed price anomalies without exposing contact fields.
price_review_path = data_dir / "suspicious_sale_prices_for_review.csv"
price_review_columns = [
    col for col in [*audit_columns, "ClosePrice", *feature_columns,
                   "OriginalListPrice", "ListPrice", "review_reason"] if col in price_review
]
price_review[price_review_columns].sort_values(
    ["ListingKeyNumeric", "source_file"], kind="stable"
).to_csv(price_review_path, index=False, date_format="%Y-%m-%d")

# Persist original values for measurement changes separately from model inputs.
measurement_changes_path = data_dir / "measurement_cleaning_audit.csv"
measurement_changes.to_csv(measurement_changes_path, index=False)

# Record exact input contents, software versions, policies, and output contents.
run_metadata = {
    "python_version": platform.python_version(),
    "package_versions": {name: version(name) for name in ["numpy", "pandas", "scikit-learn"]},
    "inputs": input_manifest,
    "notebook_sha256": file_sha256(project_root / "notebooks" / "02_preprocessing.ipynb"),
    "scope": {"state": "CA", "property_type": "Residential", "property_subtype": "SingleFamilyResidence"},
    "duplicate_policy": DUPLICATE_POLICY,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "coordinate_bounds": COORDINATE_BOUNDS,
    "missing_features": "Retained in CSV; imputation fitted on each training fold only",
    "price_rule": "Finite and positive; manually reviewed suspicious keys set aside",
    "price_review_reasons": PRICE_REVIEW_REASONS,
    "price_review_records": len(price_review),
    "price_review_sha256": file_sha256(price_review_path),
    "row_audit": row_audit,
    "review_records": len(duplicate_review),
    "measurement_rules": measurement_rule_counts,
    "measurement_screening_assumptions": MEASUREMENT_SCREENING,
    "fractional_parking_policy": "Retain fractional values; reporting convention is undocumented",
    "lot_unit_policy": "Set aside suspect measurements; no inferred conversions",
    "future_construction_policy": "Retain reviewed future years provisionally",
    "measurement_audit_sha256": file_sha256(measurement_changes_path),
    "unresolved_reviews": [
        "Additional price anomalies beyond reviewed examples", "Unit issues and extreme measurements outside explicit screens",
        "Other coordinate/city inconsistencies beyond zero/sign checks", "Prediction-time feature availability",
    ],
    "cleaned_csv_sha256": file_sha256(cleaned_csv_path),
    "duplicate_review_sha256": file_sha256(duplicate_review_path),
    "model_configuration": {
        "candidate_windows": CANDIDATE_WINDOWS, "validation_offsets": VALIDATION_OFFSETS,
        "ridge_alpha": RIDGE_ALPHA, "ridge_solver": "lsqr",
    },
}
metadata_path = data_dir / "preprocessing_metadata.json"
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")
print(f"Exported {len(cleaned_export):,} rows and {len(export_columns)} columns to {cleaned_csv_path}")
display(pd.DataFrame(row_audit))

print(f"Duplicate review records: {len(duplicate_review):,}; saved to {duplicate_review_path}")
print(f"Run metadata saved to {metadata_path}")

print(f"Suspicious price records: {len(price_review):,}; saved to {price_review_path}")


## 3. Reserve the latest month; inspect monthly coverage

A window uses calendar months immediately preceding its evaluation month. Missing months are reported and windows with gaps are skipped. A latest month with few records may be incomplete; document this, but retain the required latest-month test split.


In [ ]:
sale_month = cleaned["CloseDate"].dt.to_period("M")
test_month = sale_month.max()
# Show zero-count months explicitly. Nonzero counts do not prove a full month's data.
expected_months = pd.period_range(sale_month.min(), test_month, freq="M")
monthly_counts = sale_month.value_counts().reindex(expected_months, fill_value=0).sort_index()
print("\nMonthly sale coverage, including gaps:")
display(monthly_counts.rename("rows").to_frame())
print("Reserved test month:", test_month)
print("Latest observed sale date:", cleaned["CloseDate"].max())

def window_mask(end_month, months):
    """Select the calendar months immediately preceding an evaluation month."""
    return sale_month.ge(end_month - months) & sale_month.lt(end_month)

def has_monthly_coverage(end_month, months):
    """Check for at least one sale in each month; this does not prove complete exports."""
    expected = pd.period_range(end_month - months, end_month - 1, freq="M")
    return monthly_counts.reindex(expected, fill_value=0).gt(0).all()

print("Months with no retained sales:", [str(month) for month in monthly_counts.index[monthly_counts.eq(0)]])
print("Latest-month completeness needs confirmation from the data provider.")


## 4. Tune X on shared February and March validation months

Compare candidate training windows on the same **February and March 2026** validation months, then select the lowest mean monthly validation MAE. April 2026 remains reserved for final testing. Each training window uses only the calendar months preceding its validation month.

**Adjustment for available history:** the exports begin in November 2025. Including January as a validation month would make a three-month training window ineligible because October is unavailable. Using February and March provides enough history to compare **1- and 3-month windows** fairly:

| Validation month | 1-month training window | 3-month training window |
| --- | --- | --- |
| February 2026 | January 2026 | November 2025–January 2026 |
| March 2026 | February 2026 | December 2025–February 2026 |

Six- and twelve-month candidates remain in the configuration but are skipped for insufficient history. This adjustment is based on coverage, before inspecting validation performance. Only two validation months are available for this comparison, so the selected window may be sensitive to those months. For future exports, reassess the offsets and available history.

Imputation, encoding, scaling, and empty-column selection are fitted only on each training fold. Missing indicators and unknown-category handling support later data. Ridge with `lsqr` and chronological folds is deterministic. This selects a training window for the Ridge baseline; other models may favor different windows.


In [ ]:
def make_model(train_frame):
    """Build a deterministic Ridge baseline using features present in this training fold."""
    num = [c for c in numeric_features if train_frame[c].notna().any()]
    cat = [c for c in categorical_features if train_frame[c].notna().any()]
    transformers = []
    if num:
        transformers.append(("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), num))
    if cat:
        transformers.append(("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), cat))
    if not transformers:
        raise ValueError("No non-empty training features.")
    return Pipeline([("preprocess", ColumnTransformer(transformers)),
                     ("model", Ridge(alpha=RIDGE_ALPHA, solver="lsqr"))])

candidate_windows = CANDIDATE_WINDOWS
validation_months = [test_month - offset for offset in VALIDATION_OFFSETS
                     if (sale_month == test_month - offset).any()]
print("Shared validation months:", [str(month) for month in validation_months])
# Compare candidates on the same folds to avoid comparing different time periods.
eligible_windows = [x for x in candidate_windows if validation_months and
                    all(has_monthly_coverage(month, x) for month in validation_months)
                    and has_monthly_coverage(test_month, x)]
print("Eligible training windows (months):", eligible_windows)
print("Candidate windows lacking sufficient coverage:", sorted(set(candidate_windows) - set(eligible_windows)))
results = []
for months in eligible_windows:
    for month in validation_months:
        train_mask = window_mask(month, months)
        val_mask = sale_month.eq(month)
        model = make_model(cleaned.loc[train_mask])
        model.fit(cleaned.loc[train_mask, feature_columns], cleaned.loc[train_mask, "ClosePrice"])
        predictions = model.predict(cleaned.loc[val_mask, feature_columns])
        results.append({"training_months": months, "validation_month": str(month),
                        "train_rows": int(train_mask.sum()),
                        "validation_mae": mean_absolute_error(cleaned.loc[val_mask, "ClosePrice"], predictions)})
if not results:
    raise ValueError("Insufficient monthly coverage for these windows. Review smaller X values or validation folds.")
window_results = pd.DataFrame(results)
window_scores = window_results.groupby("training_months")["validation_mae"].mean().sort_values(kind="stable")
print("\nValidation results by training window and evaluation month:")
display(window_results)
print("\nMean validation MAE by training window (ties prefer the shorter window):")
display(window_scores.rename("mean_validation_mae").to_frame())
best_x = int(window_scores.index[0])
print("Selected training window:", best_x, "months")


## 5. Fit the selected pipeline and evaluate once on the test month

Keep the test month untouched until the window and preprocessing choices are finalized. If more changes are made after inspecting its performance, that month is no longer an untouched final test.


In [ ]:
train_mask = window_mask(test_month, best_x)
test_mask = sale_month.eq(test_month)
train = cleaned.loc[train_mask].copy()
test = cleaned.loc[test_mask].copy()
assert not train.empty and not test.empty
assert set(train["ListingKeyNumeric"]).isdisjoint(test["ListingKeyNumeric"])
assert train["CloseDate"].max() < test["CloseDate"].min()
final_pipeline = make_model(train)
final_pipeline.fit(train[feature_columns], train["ClosePrice"])
print(f"Training rows: {len(train):,}; test rows: {len(test):,}")
# Run only after preprocessing decisions are finalized:
# test_predictions = final_pipeline.predict(test[feature_columns])
# print("Test MAE:", mean_absolute_error(test["ClosePrice"], test_predictions))


## 6. Deliverable export — run after review

Export a cleaned, scoped, deduplicated table with original units, audit columns, and train/test membership. Remaining missing features are intentional: the fitted pipeline handles imputation, encoding, and normalization without leaking test information. Rows before the selected training window are labeled `outside_training_window`.

If the deliverable requires a fully encoded/imputed CSV, additionally export train/test model matrices using the training-fitted pipeline and document that they differ from the human-readable cleaned table. Preserve the fitted pipeline for reproducibility.


In [ ]:
export_frame = cleaned_export.copy()
export_frame["split"] = "outside_training_window"
export_frame.loc[train_mask, "split"] = "train"
export_frame.loc[test_mask, "split"] = "test"
export_frame["selected_training_months"] = best_x
assert export_frame["ClosePrice"].gt(0).all()
assert export_frame["ListingKeyNumeric"].is_unique
print("Proposed export:", data_dir / "cleaned_single_family_sales_with_splits.csv")
display(export_frame["split"].value_counts().rename("rows").to_frame())
# Run after selecting and reviewing the training window:
export_frame = export_frame.sort_values(["CloseDate", "ListingKeyNumeric"], kind="stable")
export_frame.to_csv(
    data_dir / "cleaned_single_family_sales_with_splits.csv", index=False, date_format="%Y-%m-%d"
)
# Extend metadata once temporal selection has run; the cleaned export works independently.
run_metadata["temporal_selection"] = {
    "test_month": str(test_month), "selected_training_months": best_x,
    "validation_results": window_results.to_dict(orient="records"),
    "train_rows": len(train), "test_rows": len(test),
}
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")
# import joblib
# joblib.dump(final_pipeline, project_root / "notebooks" / "preprocessing_pipeline.joblib")


## Session takeaways and team handoff

The baseline preprocessing deliverable was exported successfully: **60,604 California residential single-family sales** from November 2025 through April 2026. The human-readable cleaned CSV contains **17 columns**, including target and audit fields; these are not all model predictors. Original units and remaining missing features are retained. Training-fitted preprocessing handles imputation, encoding, and scaling.

The split-labeled CSV contains the same records plus `split` and `selected_training_months`:

| Split | Sale months | Rows |
| --- | --- | ---: |
| Train | January–March 2026 | 27,974 |
| Test | April 2026 | 12,480 |
| Outside selected training window | November–December 2025 | 20,150 |
| Total | November 2025–April 2026 | 60,604 |

Earlier rows remain in the export for traceability and were used in some validation folds. They are excluded from the final three-month model fit; they are not rejected as poor-quality records.

### Baseline results

Shared February/March validation folds selected a **three-month training window** for Ridge. Mean monthly validation MAE was **$331,603.68**, compared with **$355,066.26** for one month, an improvement of approximately **6.6%**. Six- and twelve-month windows were ineligible because the source history was too short.

The fitted January–March model achieved **April test MAE of $354,694.32** across **12,480 sales**, approximately 7% higher than validation MAE. MAE is average absolute dollar error, not an error bound for each home. These results establish a baseline; they do not establish reliable accuracy for every California address.

### Cleaning decisions and artifacts

- Scope filtering retained 60,694 California residential single-family records from 122,756 imported records.
- Set aside all 79 records belonging to 39 conflicting listing keys, and 11 individually reviewed suspicious-price records. No sale-price replacements were inferred.
- Converted nonpositive areas, negative counts, the 17 observed invalid coordinate pairs, and measurements failing explicit provisional screens to missing. Property rows were retained through measurement cleaning.
- Preserved fractional garage/parking values without rounding and the reviewed future construction years provisionally.
- Exported `data/cleaned_single_family_sales.csv`, `data/cleaned_single_family_sales_with_splits.csv`, duplicate and price review CSVs, `data/measurement_cleaning_audit.csv`, and `data/preprocessing_metadata.json`. Keep the audits and metadata with the deliverable. The fitted pipeline export remains optional/commented in the notebook; CSVs alone do not preserve learned medians, category mappings, or model coefficients.

### Notes for the team

1. **Prediction-time inputs:** establish how property characteristics and location will be obtained for an off-market address. Sold-listing values are not proof that those same values were available at an earlier query time. Listing prices used during quality review remain excluded from predictors.
2. **Source documentation:** ask the data provider to clarify repeated listing keys, fractional parking conventions, lot units, suspicious sale prices, and whether monthly exports are complete. Restore quarantined records or measurements only after verification.
3. **Provisional screens:** garage values above 20, parking above 50, positive lots below 1 square foot, and the documented suspected double-conversion pattern were treated as unverified. These are baseline assumptions, not physical limits. They can mask legitimate measurements and do not identify all source errors. Future construction years also remain uncertain.
4. **Location validation:** zero/sign checks do not confirm California boundary membership or agreement with city/ZIP/address. Some plausible-looking coordinates may still be wrong.
5. **Evaluation limits:** only two months informed window selection. April was excluded from model tuning, but April records were inspected for quality and its model score has now been viewed. Use earlier rolling validation for further development; obtain a new untouched holdout for a subsequent final claim.
6. **Generalization:** the evaluated population consists of sold homes. Accuracy on off-market homes, future months, different geographic areas, and price segments remains unverified. Consider median absolute error and errors by price/location during future validation to complement overall MAE.
7. **Reproducibility:** save the notebook and preserve exact source files, hashes, package versions, configuration, and audits. If handing off predictions, persist the fitted pipeline as well. The current temporal metadata records the window/fold results and split counts; the reported test MAE is documented here but is not automatically stored by the existing export cell. Do not rerun and retune against April's score.

These figures summarize the outputs reviewed in this session. Update this handoff if inputs, policies, features, or model results change. Remaining uncertainties are documented limitations of the baseline deliverable.
